# Bibliotecas necessárias

In [1]:
import sys
import os
from pyspark.sql import SparkSession
sys.path.append('/home/jovyan/work')
from datetime import datetime, timedelta
from _teste_crawler_sti import create_df,insert_datalake

tabela = 'sti_atendimentos_sgt'
endpoint ='https://apisgt.cni.org.br:8253/reports/v1.0.0/atendimentos'

# Cricação da variáveis necessárias

In [2]:
#Último x dias
#dias = 5
#start_date = datetime.now() - timedelta(days=dias)
#date = start_date.strftime('%Y%m%d')

# Data Atul
#date_now = datetime.now().strftime('%Y%m%d')

#Data específica
date = '20230101'

# Extração

In [3]:
#df = create_df(endpoint, date, tabela)

/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apisgt.cni.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


DataFrame criado para sti_atendimentos_sgt na data 2024-05-07


In [10]:
import requests

B_TOKEN = "Bearer 48152c5b-3d82-3cba-a836-931698063f44"

payload = ""
headers = {
    "accept": "application/json",
    "authorization": B_TOKEN
}

querystring = {"dataInicio": date}

response = requests.request("GET", endpoint, data=payload, headers=headers, params=querystring, verify=False)

data = response.json()

/opt/conda/lib/python3.11/site-packages/urllib3/connectionpool.py:1100: InsecureRequestWarning: Unverified HTTPS request is being made to host 'apisgt.cni.org.br'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [11]:
from pyspark.sql.types import StructType, StructField, StringType

spark = SparkSession.builder \
     .appName("Exemplo de DataFrame Spark") \
     .getOrCreate()

schema_sti_atendimentos_sgt = StructType([
            StructField("id", StringType(), True),
            StructField("idAtendimento", StringType(), True),
            StructField("numeroAtendimento", StringType(), True),
            StructField("tituloAtendimento", StringType(), True),
            StructField("descricaoStatusAtendimento", StringType(), True),
            StructField("dia", StringType(), True),
            StructField("mes", StringType(), True),
            StructField("ano", StringType(), True),
            StructField("cpfCnpjCliente", StringType(), True),
            StructField("nomeCliente", StringType(), True),
            StructField("idPorteCliente", StringType(), True),
            StructField("porteCliente", StringType(), True),
            StructField("numeroDeFuncionarios", StringType(), True),
            StructField("descricaoProdutoRegional", StringType(), True),
            StructField("CRProdutoRegional", StringType(), True),
            StructField("codigoDNProdutoNacional", StringType(), True),
            StructField("descricaoProdutoNacional", StringType(), True),
            StructField("codigoDNProdutoCategoria", StringType(), True),
            StructField("descricaoProdutoCategoria", StringType(), True),
            StructField("codigoDNProdutoLinha", StringType(), True),
            StructField("descricaoProdutoLinha", StringType(), True),
            StructField("codigoOBAUnidadeOperacional", StringType(), True),
            StructField("descricaoUnidadeOperacional", StringType(), True),
            StructField("numeroDeProducaoEstimada", StringType(), True),
            StructField("numeroDeRelatorioEstimado", StringType(), True),
            StructField("numeroDeReceitaEstimada", StringType(), True),
            StructField("vlrFinanceiro", StringType(), True),
            StructField("vlrEconomico", StringType(), True),
            StructField("isCompartilhado", StringType(), True),
            StructField("isEmRede", StringType(), True),
            StructField("isEscopoIndefinido", StringType(), True),
            StructField("isValorHora", StringType(), True)
            ])


def process_row(row):
    if 'numeroAtendimento' in row and isinstance(row['numeroAtendimento'], dict) and '@nil' in row['numeroAtendimento']:
        row['numeroAtendimento'] = row['numeroAtendimento']['@nil']
    if 'numeroDeFuncionarios' in row and isinstance(row['numeroDeFuncionarios'], dict) and '@nil' in row['numeroDeFuncionarios']:
        row['numeroDeFuncionarios'] = row['numeroDeFuncionarios']['@nil']
    if 'isFonteFomento' in row and isinstance(row['isFonteFomento'], dict) and '@nil' in row['isFonteFomento']:
        row['isFonteFomento'] = row['isFonteFomento']['@nil']
    if 'incricaoEstadual' in row and isinstance(row['incricaoEstadual'], dict) and '@nil' in row['incricaoEstadual']:
        row['incricaoEstadual'] = row['incricaoEstadual']['@nil']
    if 'idPorteCliente' in row and isinstance(row['idPorteCliente'], dict) and '@nil' in row['idPorteCliente']:
        row['idPorteCliente'] = row['idPorteCliente']['@nil']
    if 'porteCliente' in row and isinstance(row['porteCliente'], dict) and '@nil' in row['porteCliente']:
        row['porteCliente'] = row['porteCliente']['@nil']
    if 'nomeColaborador' in row and isinstance(row['nomeColaborador'], dict) and '@nil' in row['nomeColaborador']:
        row['nomeColaborador'] = row['nomeColaborador']['@nil']
    if 'nomeFontePagadora' in row and isinstance(row['nomeFontePagadora'], dict) and '@nil' in row['nomeFontePagadora']:
        row['nomeFontePagadora'] = row['nomeFontePagadora']['@nil']
    if 'valorReceitaRealizada' in row and isinstance(row['valorReceitaRealizada'], dict) and '@nil' in row['valorReceitaRealizada']:
        row['valorReceitaRealizada'] = row['valorReceitaRealizada']['@nil']
    return row


data = data['Atendimentos']['Atendimento']

In [12]:
processed_data = map(process_row, data)

df_spark = spark.createDataFrame(processed_data, schema=schema_sti_atendimentos_sgt)

In [13]:
df_spark.select('numeroAtendimento', 'numeroDeFuncionarios').show(10)

+-----------------+--------------------+
|numeroAtendimento|numeroDeFuncionarios|
+-----------------+--------------------+
|           181281|                  10|
|           190306|                  10|
|           190428|                  10|
|           190500|                  10|
|           190580|                  10|
|           190663|                  10|
|           190803|                  10|
|           210281|                  10|
|           210396|                   0|
|           210434|                   0|
+-----------------+--------------------+
only showing top 10 rows



# Gravação no datalake

In [ ]:
#insert_datalake(df, tabela)